# Notebook 5: Reading & Writing Data
**Filename:** `05_File_Operations.ipynb`  
**Topic:** File I/O Operations across CSV, Excel, JSON, Parquet, Pickle, SQL, Multi-file Loading, and Path Handling

---

## 1. CSV Files

### Concept Explanation
Comma-Separated Values (CSV) is a plain-text tabular format. `pd.read_csv()` and `df.to_csv()` provide flexible parameters like `sep` (custom delimiters), `usecols` (selecting subset columns), `skiprows`, and `encoding` to handle varied text datasets.

### Real-world Example
Loading web server access logs exported as tab-separated or comma-separated log text files.

### Business Example
Importing massive monthly retail transaction dumps while skipping header commentary rows and loading only key financial columns to save memory.

### AI/ML Example
Loading high-dimensional tabular datasets for feature extraction while specifying custom missing value tokens (`na_values`).

In [1]:
import pandas as pd

# Creating sample CSV data with custom separator and extra header comments
csv_content = """# Export Date: 2026-08-19
# System: POS Terminal
ID|Product|Price
101|Laptop|1200
102|Mouse|25
103|Keyboard|45"""

with open("transactions.csv", "w") as f:
    f.write(csv_content)

# Reading CSV skipping comment rows and selecting specific columns
df_csv = pd.read_csv(
    "transactions.csv", 
    sep="|", 
    skiprows=2, 
    usecols=["Product", "Price"]
)

print("Parsed CSV Data:\n", df_csv)

# Writing back to CSV with custom delimiter
df_csv.to_csv("processed_transactions.csv", sep=";", index=False)

Parsed CSV Data:
     Product  Price
0    Laptop   1200
1     Mouse     25
2  Keyboard     45


---

## 2. Excel Files

### Concept Explanation
`pd.read_excel()` and `df.to_excel()` interact with Microsoft Excel workbooks (`.xlsx`), enabling reading from or writing to specific sheets using parameter `sheet_name` via openpyxl.

### Real-world Example
Extracting monthly expense summaries from multi-tab Excel workbooks shared by non-technical administrative teams.

### Business Example
Automating quarterly financial consolidations across dozens of department-specific budget workbooks.

### AI/ML Example
Ingesting manually annotated domain expert datasets stored as multi-sheet spreadsheets into python feature pipelines.

In [2]:
import pandas as pd

df_sales = pd.DataFrame({'Region': ['North', 'South'], 'Revenue': [50000, 62000]})
df_targets = pd.DataFrame({'Region': ['North', 'South'], 'Target': [45000, 60000]})

# Writing multiple DataFrames to separate sheets in one Excel file
with pd.ExcelWriter("company_report.xlsx", engine="openpyxl") as writer:
    df_sales.to_excel(writer, sheet_name="Sales", index=False)
    df_targets.to_excel(writer, sheet_name="Targets", index=False)

# Reading a specific sheet from Excel
df_read_sales = pd.read_excel("company_report.xlsx", sheet_name="Sales")
print("Data read from Excel Sheet 'Sales':\n", df_read_sales)

ModuleNotFoundError: No module named 'openpyxl'

---

## 3. JSON Files

### Concept Explanation
JavaScript Object Notation (JSON) is a lightweight format popular in web APIs. `pd.read_json()` and `df.to_json()` handle semi-structured data using orientations like `records`, `split`, or `orient='index'`.

### Real-world Example
Ingesting real-time weather observation payloads sent by REST API endpoints.

### Business Example
Parsing multi-nested customer profile objects received from a web application backend.

### AI/ML Example
Converting unstructured model configuration logs or prediction output payloads to tabular format for analysis.

In [3]:
import pandas as pd

df = pd.DataFrame({
    'user_id': [1, 2],
    'settings': [{'theme': 'dark'}, {'theme': 'light'}]
})

# Exporting DataFrame to JSON (records orient)
df.to_json("users.json", orient="records", indent=2)

# Reading JSON file back into Pandas
df_json = pd.read_json("users.json", orient="records")
print("Parsed JSON Data:\n", df_json)

Parsed JSON Data:
    user_id            settings
0        1   {'theme': 'dark'}
1        2  {'theme': 'light'}


---

## 4. Parquet Files

### Concept Explanation
Parquet is an open-source, column-oriented binary storage format designed for fast analytics, high compression ratios, and efficient file I/O operations compared to row-based formats like CSV.

### Real-world Example
Storing terabytes of IoT sensor logs where querying specific individual columns requires reading only a fraction of total disk data.

### Business Example
Saving massive transactional data archives in enterprise cloud data lakes (AWS S3 / Azure Data Lake) to minimize storage costs and accelerate query performance.

### AI/ML Example
Reading and writing compressed feature stores during large-scale distributed ML model training (e.g., PySpark or Dask integrations).

In [4]:
import pandas as pd

# Create sample large dataset
df_large = pd.DataFrame({
    'sensor_id': range(1000),
    'reading': [22.5 * i for i in range(1000)]
})

# Write to Parquet binary format (requires pyarrow or fastparquet)
df_large.to_parquet("sensor_data.parquet", index=False)

# Read Parquet file
df_parquet = pd.read_parquet("sensor_data.parquet")
print(f"Parquet Read Successfully! Shape: {df_parquet.shape}")

Parquet Read Successfully! Shape: (1000, 2)


---

## 5. Pickle Files

### Concept Explanation
Pickling is Python’s native object serialization format. `df.to_pickle()` and `pd.read_pickle()` preserve full Python and Pandas data types, categorical encodings, and index structures without data loss or type re-parsing.

### Real-world Example
Temporarily saving intermediate data processing steps during a local Python script execution session.

### Business Example
Saving preprocessed master datasets between daily scheduled analytics script runs without losing custom index formatting.

### AI/ML Example
Caching cleaned, feature-engineered DataFrames to disk so model training scripts re-load preprocessed data instantaneously without re-running data cleaning steps.

In [5]:
import pandas as pd

df_original = pd.DataFrame({
    'Category': pd.Series(['A', 'B', 'A'], dtype='category'),
    'Timestamp': pd.to_datetime(['2026-08-01', '2026-08-02', '2026-08-03'])
})

# Save state completely using pickle
df_original.to_pickle("preprocessed_data.pkl")

# Reload pickle preserving exact data types
df_pickled = pd.read_pickle("preprocessed_data.pkl")
print("Pickled Data Types preserved exactly:\n", df_pickled.dtypes)

Pickled Data Types preserved exactly:
 Category           category
Timestamp    datetime64[us]
dtype: object


---

## 6. SQL Tables

### Concept Explanation
Pandas interacts directly with Relational Database Management Systems (RDBMS) like SQLite, PostgreSQL, or MySQL via SQLAlchemy using `pd.read_sql()` and `df.to_sql()`.

### Real-world Example
Connecting to a local SQLite database file to pull live customer records.

### Business Example
Writing aggregated daily revenue metrics directly back into enterprise SQL database tables for executive dashboards.

### AI/ML Example
Executing SQL query filters (`SELECT * FROM table WHERE active=1`) to load specific database cohorts into Pandas feature matrices.

In [6]:
import pandas as pd
from sqlalchemy import create_engine

# Create an in-memory SQLite database connection
engine = create_engine('sqlite:///:memory:')

# Sample DataFrame
df_users = pd.DataFrame({
    'user_id': [101, 102, 103],
    'username': ['alpha', 'beta', 'gamma']
})

# Write DataFrame to SQL Table
df_users.to_sql('users', con=engine, index=False, if_exists='replace')

# Read from SQL using a Query
df_sql = pd.read_sql("SELECT * FROM users WHERE user_id > 101", con=engine)
print("Queried SQL Result:\n", df_sql)

Queried SQL Result:
    user_id username
0      102     beta
1      103    gamma


---

## 7. Reading Multiple Files

### Concept Explanation
When datasets are split across multiple files (e.g., daily CSV reports), the standard `glob` module finds matching file paths, and `pd.concat()` combines them into a unified single DataFrame.

### Real-world Example
Combining 12 separate monthly CSV files into a single master annual report.

### Business Example
Merging daily log files generated by distributed regional servers into one centralized analytics table.

### AI/ML Example
Aggregating partitioned dataset chunks exported from cloud storage before running global data preprocessing pipelines.

In [7]:
import pandas as pd
import glob

# Creating dummy daily files
pd.DataFrame({'day': [1], 'sales': [100]}).to_csv("day_1.csv", index=False)
pd.DataFrame({'day': [2], 'sales': [150]}).to_csv("day_2.csv", index=False)

# Finding all matching file paths
file_pattern = "day_*.csv"
file_list = glob.glob(file_pattern)

# Reading and concatenating into one DataFrame
df_combined = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)
print("Combined Multi-file DataFrame:\n", df_combined)

Combined Multi-file DataFrame:
    day  sales
0    1    100
1    2    150


---

## 8. File Path Handling

### Concept Explanation
Hardcoding file path strings causes platform errors across Windows (`\`), macOS, and Linux (`/`). The native `pathlib.Path` library manages cross-platform paths, directory creation, and file existence checks cleanly.

### Real-world Example
Writing cross-platform Python scripts that run seamlessly on developer MacBooks and Windows servers.

### Business Example
Ensuring automated data ingestion pipelines safely check for source file existence before attempting to open them.

### AI/ML Example
Constructing dynamic output folder structures to save trained model artifacts, evaluation metrics, and prediction plots cleanly.

In [8]:
from pathlib import Path
import pandas as pd

# Creating cross-platform directory structure using pathlib
data_dir = Path("data_folder")
data_dir.mkdir(exist_ok=True)  # Create directory if it doesn't exist

file_path = data_dir / "sample_output.csv"

# Write DataFrame using Path object
df = pd.DataFrame({'status': ['OK', 'PENDING']})
df.to_csv(file_path, index=False)

# Check existence and read
if file_path.exists():
    df_read = pd.read_csv(file_path)
    print("Successfully read file from Path object:\n", df_read)

Successfully read file from Path object:
     status
0       OK
1  PENDING


---

## Minimum 5 Interview Questions with Answers

1. **Why is Apache Parquet preferred over CSV for storing large analytical datasets?**  
   * **Answer:** Parquet is a compressed columnar binary format. It reduces disk storage overhead significantly, preserves data types natively, and allows query engines to read only required columns rather than scanning entire rows.

2. **How does `to_pickle()` differ from exporting to `to_csv()`?**  
   * **Answer:** `to_pickle()` serializes Python/Pandas objects natively, retaining explicit data types (like categoricals, datetime zones, and complex index structures). `to_csv()` converts all data to plain text, requiring type re-parsing upon reload.

3. **What is the purpose of using `if_exists='replace'` or `'append'` in `df.to_sql()`?**  
   * **Answer:** The `if_exists` parameter controls database behavior when the destination table already exists: `'replace'` drops and recreates the table, `'append'` inserts new rows into the existing table, and `'fail'` raises an error.

4. **How do you efficiently combine hundreds of CSV files with identical columns into one DataFrame?**  
   * **Answer:** Use Python's `glob` module to retrieve matching file paths, generate a list of DataFrames using a list comprehension with `pd.read_csv()`, and combine them with `pd.concat(list_of_dfs, ignore_index=True)`.

5. **Why should you use `pathlib.Path` instead of raw string paths like `"C:\\data\\file.csv"`?**  
   * **Answer:** `pathlib.Path` provides object-oriented, cross-platform path handling that resolves operating system path separator differences (`\` vs `/`) and offers safe built-in utilities for checking file existence and directory management.

---

## Self Reflection
* **What I Learned:** Learned file I/O operations across structured text (CSV, JSON), spreadsheets (Excel), high-performance formats (Parquet, Pickle), relational databases (SQL), batch multi-file processing, and robust cross-platform path handling.
* **Key Takeaway:** Choosing the right file storage format (e.g., Parquet for scale, Pickle for intermediate caching, CSV/Excel for human exchange) drastically impacts memory usage, processing speed, and pipeline reliability.